# Express Links · Step 2b + 12b: fix the missing data, retrain, compare (METR-LA)

**Question this notebook answers:** did the way step 2 handled broken sensors cause the tie between express and random links in step 12?

**What it does, top to bottom**

| Part | What | GPU? | Time |
|---|---|---|---|
| A (sections 1–6) | **Fixes the data.** Broken-sensor zeros in the model's **inputs** are filled (short gaps: last real reading; long gaps: the sensor's usual speed at that time of week, from training data only). The **targets are not changed**: missing ones stay 0 and are skipped in scoring, as before. Writes to `data/processed_v2/METR-LA/`. Your step 2 files are not touched. | no | ~5 min |
| B (sections 7–9) | **Retrains** Graph WaveNet on the fixed data with 3 graph versions: `original`, `random`, `express`. Same model, settings and 100 epochs as step 12, so the only change is the data. Runs go to `runs_v2/`; your old `runs/` are not touched. | yes | see below |
| C (section 10) | **Compares** old data vs fixed data, scored with the same step-9 code on the same test set, and prints the answer. | no | ~2 min |

**GPU time per run (100 epochs):** about 4 h on a Colab T4, about 1 h on an A100, about 25 min on the lab RTX PRO 6000. Default is **seed 0 only = 3 runs**. Add seeds in section 7 if the first answer is close.

**Colab disconnects are fine.** Every epoch is saved to Drive. Just **Run all** again: part A skips itself if the files are already there, and part B continues from the last saved epoch.

**Before you start:** Runtime → Change runtime type → **GPU**. Then Runtime → Run all.

## 1. Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
FOLDER = os.path.join('AdvML_F26_Project_&_Survey_Paper', 'Project', 'Coding')
ROOTS = ['/content/drive/MyDrive', '/content/drive/.shortcut-targets-by-id', '/content/drive/Shareddrives']
hits = []
for root in ROOTS:
    for depth in range(4):
        hits += glob.glob(os.path.join(root, *(['*'] * depth), FOLDER))
hits = sorted(set(h for h in hits if os.path.isdir(os.path.join(h, 'data'))), key=len)
assert hits, "Couldn't find the 'Coding' folder. Add a shortcut to 'AdvML_F26_Project_&_Survey_Paper' in My Drive, then rerun."
PROJECT_DIR = hits[0]
print('PROJECT_DIR =', PROJECT_DIR)

Mounted at /content/drive
PROJECT_DIR = /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding


## 2. Settings

* `SHORT_GAP` – gaps up to this many steps are filled by carrying the last real reading forward (3 = 15 min).
* `STUCK_STEPS` – a sensor repeating the exact same non-zero value this many steps in a row counts as stuck (12 = 1 hour).
* `STUCK_AS_MISSING` – `False` = only count stuck stretches. `True` = treat them as missing in the **inputs** (targets are never changed). Look at the audit in section 4 before switching it on.
* `DEAD_SHARE` – a sensor missing more than this share of the test period is listed as dead.
* `ADD_MASK_CHANNEL` – keep `False`: `x` keeps 2 channels (speed, time of day), so `train_gwnet.py` runs unchanged. The 0/1 mask is saved as a separate array `x_mask`.

In [2]:
import os, json, time, pickle, gc
import numpy as np
import pandas as pd

RAW_DIR  = os.path.join(PROJECT_DIR, 'data', 'raw')
V1_ROOT  = os.path.join(PROJECT_DIR, 'data', 'processed')        # step 2 (read only)
OUT_ROOT = os.path.join(PROJECT_DIR, 'data', 'processed_v2')     # this notebook writes here
os.makedirs(OUT_ROOT, exist_ok=True)

ALL_DATASETS = {'METR-LA': ('metr-la.h5', 'adj_mx.pkl'), 'PEMS-BAY': ('pems-bay.h5', 'adj_mx_bay.pkl')}
DATASETS = {k: ALL_DATASETS[k] for k in ['METR-LA']}   # PEMS-BAY has almost no missing data, so it is left out
REBUILD = False            # True = rebuild the fixed files even if they already exist
SEQ_IN, SEQ_OUT = 12, 12
TRAIN_FRAC, TEST_FRAC = 0.7, 0.2

SHORT_GAP        = 3
STUCK_STEPS      = 12
STUCK_AS_MISSING = False
DEAD_SHARE       = 0.5
ADD_MASK_CHANNEL = False
print('writing to', OUT_ROOT)

writing to /content/drive/MyDrive/AdvML_F26_Project_&_Survey_Paper/Project/Coding/data/processed_v2


## 3. The functions

Each function has one job. Section 3b tests them on a small made-up table where the right answer is known, before they touch the real data.

In [3]:
def regrid(df):
    """Put the table on a full 5-minute clock. Missing timestamps become rows of 0 (= missing)."""
    idx = pd.DatetimeIndex(np.asarray(df.index.values, dtype='datetime64[ns]'))
    df = df.copy(); df.index = idx
    full = pd.date_range(idx[0], idx[-1], freq='5min')
    added = full.difference(idx)
    return df.reindex(full, fill_value=0.0), added


def run_lengths(mask_col):
    """For a 1-D bool array, the length of the True-run each element belongs to (0 where False)."""
    m = np.asarray(mask_col, bool)
    out = np.zeros(len(m), int)
    if not m.any():
        return out
    d = np.diff(np.concatenate([[0], m.astype(int), [0]]))
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    for s, e in zip(starts, ends):
        out[s:e] = e - s
    return out


def stuck_mask(values, min_steps=STUCK_STEPS):
    """(T, N) True where a sensor repeats the same non-zero value for >= min_steps steps."""
    T, N = values.shape
    same = np.vstack([np.zeros((1, N), bool), (values[1:] == values[:-1]) & (values[1:] > 0)])
    out = np.zeros((T, N), bool)
    for n in range(N):
        # a run of k 'same as previous' flags means k+1 identical readings
        rl = run_lengths(same[:, n])
        hit = rl + 1 >= min_steps
        idx = np.flatnonzero(hit)
        out[idx, n] = True
        out[np.clip(idx - 1, 0, T - 1), n] |= True           # include the first reading of the run
    return out


def steps_since_valid(observed):
    """(T, N): number of steps since the last observed reading (0 when observed now). Causal."""
    T, N = observed.shape
    out = np.zeros((T, N), int)
    last = np.full(N, -10**9)
    for t in range(T):
        last = np.where(observed[t], t, last)
        out[t] = t - last
    return out


def history_profile(values, observed, index, train_last_row):
    """Each sensor's usual speed by time of week, from training rows only (rows <= train_last_row).
    Falls back to time of day, then to the sensor's overall training mean."""
    tr = slice(0, train_last_row + 1)
    v = np.where(observed[tr], values[tr], np.nan)
    tod  = (index.hour * 12 + index.minute // 5).values               # 0..287
    tow  = index.dayofweek.values * 288 + tod                           # 0..2015
    df_v = pd.DataFrame(v)
    by_tow = df_v.groupby(tow[tr]).mean().reindex(range(7 * 288))
    by_tod = df_v.groupby(tod[tr]).mean().reindex(range(288))
    overall = np.nanmean(v, axis=0)
    prof_tow = by_tow.values                                            # (2016, N)
    prof_tod = by_tod.values                                            # (288, N)
    hist = prof_tow[tow]
    hist = np.where(np.isnan(hist), prof_tod[tod], hist)
    hist = np.where(np.isnan(hist), overall[None, :], hist)
    return hist.astype(np.float32)


def fill_inputs(values, observed, hist, short_gap=SHORT_GAP):
    """Causal fill. Within short_gap steps of the last real reading: carry it forward.
    Longer: the training-period profile for that sensor and time of week."""
    sv = steps_since_valid(observed)
    ffill = pd.DataFrame(np.where(observed, values, np.nan)).ffill().values
    filled = values.astype(np.float32).copy()
    short = (~observed) & (sv <= short_gap) & ~np.isnan(ffill)
    long_ = (~observed) & ~short
    filled[short] = ffill[short]
    filled[long_] = hist[long_]
    return filled, short, long_


def build_samples(inp, tgt, obs, tod, seq_in=SEQ_IN, seq_out=SEQ_OUT, add_mask=ADD_MASK_CHANNEL):
    """Same anchors as Graph WaveNet / step 2. x from the filled inputs, y from the untouched targets."""
    T, N = inp.shape
    tod_b = np.broadcast_to(tod[:, None], (T, N))
    chans = [inp, tod_b] + ([obs.astype(np.float32)] if add_mask else [])
    xdata = np.stack(chans, axis=-1).astype(np.float32)
    ydata = np.stack([tgt, tod_b], axis=-1).astype(np.float32)
    anchors = np.arange(seq_in - 1, T - seq_out)
    x_off, y_off = np.arange(-(seq_in - 1), 1), np.arange(1, seq_out + 1)
    x = xdata[anchors[:, None] + x_off[None, :]]
    y = ydata[anchors[:, None] + y_off[None, :]]
    xm = obs[anchors[:, None] + x_off[None, :]]
    return x, y, xm, anchors, x_off, y_off


def split_sizes(n, train_frac=TRAIN_FRAC, test_frac=TEST_FRAC):
    n_test, n_train = round(n * test_frac), round(n * train_frac)
    return n_train, n - n_test - n_train, n_test

### 3b. Test the functions on a made-up table (right answers known)

In [4]:
idx = pd.date_range('2020-01-06', periods=288 * 14, freq='5min')     # two weeks, starts on a Monday
rng = np.random.default_rng(0)
true = 60 + 5 * np.sin(np.arange(len(idx)) / 288 * 2 * np.pi)[:, None] + rng.normal(0, 1, (len(idx), 2))
vals = true.copy()
vals[100:102, 0] = 0          # 2-step gap   -> carry forward
vals[300:340, 0] = 0          # 40-step gap  -> profile for the first 3 steps? no: carry 3, then profile
vals[500:520, 1] = 61.0       # stuck for 20 steps
toy = pd.DataFrame(vals, index=idx).drop(idx[1000:1012])              # a lost hour

g, added = regrid(toy)
assert len(added) == 12 and (g.loc[added].values == 0).all();                print('OK  regrid adds the lost hour as missing')
obs = g.values > 0
assert stuck_mask(g.values)[500:520, 1].all() and not stuck_mask(g.values)[:, 0].any(); print('OK  stuck stretch found, nothing else')
ss = steps_since_valid(obs)
assert ss[100, 0] == 1 and ss[101, 0] == 2 and ss[339, 0] == 40 and ss[340, 0] == 0;    print('OK  steps since last real reading')
hist = history_profile(g.values, obs, g.index, train_last_row=288 * 10)
filled, short, long_ = fill_inputs(g.values, obs, hist)
assert np.isclose(filled[100, 0], g.values[99, 0]) and np.isclose(filled[101, 0], g.values[99, 0]);       print('OK  short gap carries the last real reading')
assert np.isclose(filled[302, 0], g.values[299, 0]) and np.isclose(filled[303, 0], hist[303, 0]);         print('OK  long gap: 3 steps carried, then the training profile')
assert (filled > 0).all();                                                              print('OK  no zeros left in the inputs')
# causality: changing the future must not change any earlier filled value
g2 = g.values.copy(); g2[3000:] = 10.0
f2, _, _ = fill_inputs(g2, g2 > 0, hist)
assert np.allclose(f2[:3000], filled[:3000]);                                          print('OK  filling never looks into the future')

OK  regrid adds the lost hour as missing
OK  stuck stretch found, nothing else
OK  steps since last real reading
OK  short gap carries the last real reading
OK  long gap: 3 steps carried, then the training profile
OK  no zeros left in the inputs
OK  filling never looks into the future


# Part A: fix the data
## 4. Load the raw files and audit every scenario

In [5]:
def load_raw(name):
    h5, adj = DATASETS[name]
    df = pd.read_hdf(os.path.join(RAW_DIR, h5)).sort_index()
    with open(os.path.join(RAW_DIR, adj), 'rb') as f:
        graph_ids = [str(s) for s in pickle.load(f, encoding='latin1')[0]]
    return df, graph_ids


def audit(name, df, graph_ids, g, added):
    v = g.values; T, N = v.shape
    miss = v == 0
    v1 = json.load(open(os.path.join(V1_ROOT, name, 'summary.json')))
    n_tr, n_va, n_te = split_sizes(T - SEQ_IN - SEQ_OUT + 1)
    test_first_row = n_tr + n_va + SEQ_IN - 1
    rl = np.stack([run_lengths(miss[:, n]) for n in range(N)], axis=1)
    per_t = miss.mean(axis=1)
    stuck = stuck_mask(v)
    dead = miss[test_first_row:].mean(axis=0) > DEAD_SHARE
    rows = [
        ('sensor order of data matches road graph', [str(c) for c in df.columns] == graph_ids, 'checked', 'checked'),
        ('missing timestamps (re-gridded)', f'{len(added)} rows' + (f' ({added[0]} …)' if len(added) else ''), 'no', 'yes'),
        ('readings missing (all)', f'{miss.mean():.2%}', 'masked in loss only', 'inputs filled + mask'),
        ('  in gaps <= 15 min', f'{(miss & (rl <= SHORT_GAP)).mean():.2%}', 'no', 'carry forward'),
        ('  in gaps 15 min - 2 h', f'{(miss & (rl > SHORT_GAP) & (rl <= 24)).mean():.2%}', 'no', 'profile'),
        ('  in gaps > 2 h', f'{(miss & (rl > 24)).mean():.2%}', 'no', 'profile'),
        ('timestamps with > 50% of sensors off', f'{(per_t > 0.5).sum():,}', 'no', 'profile; reported'),
        ('stuck readings (same value >= 1 h)', f'{stuck.mean():.2%}', 'no', 'counted' + (' + treated as missing' if STUCK_AS_MISSING else '')),
        (f'dead sensors in test (> {DEAD_SHARE:.0%} missing)', f'{int(dead.sum())}', 'no', 'listed in sensor_quality.csv'),
        ('missing targets: train / val / test',
         ' / '.join(f"{v1['splits'][p]['masked_share_targets']:.1%}" for p in ['train', 'val', 'test']), 'not reported', 'reported'),
        ('scaling mean with zeros / without',
         f"{np.mean(v[:n_tr + SEQ_IN - 1]):.2f} / {v[:n_tr + SEQ_IN - 1][v[:n_tr + SEQ_IN - 1] > 0].mean():.2f} mph", 'with zeros', 'without'),
    ]
    tab = pd.DataFrame(rows, columns=['scenario', 'found', 'step 2', 'step 2b']).set_index('scenario')
    quality = pd.DataFrame({'sensor_id': [str(c) for c in df.columns],
                            'missing_all': miss.mean(axis=0),
                            'missing_test': miss[test_first_row:].mean(axis=0),
                            'longest_gap_hours': rl.max(axis=0) * 5 / 60,
                            'stuck_share': stuck.mean(axis=0),
                            'dead_in_test': dead})
    return tab, quality


raw, audits = {}, {}
for name in DATASETS:
    df, graph_ids = load_raw(name)
    g, added = regrid(df)
    raw[name] = (df, graph_ids, g, added)
    tab, quality = audit(name, df, graph_ids, g, added)
    audits[name] = (tab, quality)
    os.makedirs(os.path.join(OUT_ROOT, name), exist_ok=True)
    quality.to_csv(os.path.join(OUT_ROOT, name, 'sensor_quality.csv'), index=False)
    print(f'==== {name} ====')
    display(tab)
    worst = quality.sort_values('missing_test', ascending=False).head(8)
    print('Worst sensors in the test period:'); display(worst.round(3))

==== METR-LA ====


,found,step 2,step 2b
scenario,,,
sensor order of data matches road graph,True,checked,checked
missing timestamps (re-gridded),0 rows,no,yes
readings missing (all),8.11%,masked in loss only,inputs filled + mask
in gaps <= 15 min,0.34%,no,carry forward
in gaps 15 min - 2 h,0.97%,no,profile
in gaps > 2 h,6.80%,no,profile
timestamps with > 50% of sensors off,"2,149",no,profile; reported
stuck readings (same value >= 1 h),0.05%,no,counted
dead sensors in test (> 50% missing),0,no,listed in sensor_quality.csv


Worst sensors in the test period:


,sensor_id,missing_all,missing_test,longest_gap_hours,stuck_share,dead_in_test
56,716955,0.157,0.444,71.417,0.0,False
82,769430,0.119,0.369,110.833,0.0,False
70,717490,0.118,0.319,140.250,0.0,False
71,717491,0.117,0.318,140.250,0.0,False
60,769444,0.107,0.309,110.833,0.0,False
81,769431,0.107,0.309,110.833,0.0,False
63,769443,0.107,0.309,110.833,0.0,False
180,717510,0.150,0.257,31.750,0.0,False


**How to read the audit.** METR-LA is where missing data lives (about 8% of readings, and much more in the test period). PEMS-BAY has almost none (about 0.003%). Caltrans PeMS **already fills its own gaps** before publishing, so PEMS-BAY's real outages are hidden as plausible-looking values, not zeros. The stuck-reading count is the only trace of that, so look at it before switching on `STUCK_AS_MISSING`.

## 5. Build and save the fixed files (`data/processed_v2/<dataset>/`)

In [6]:
def prepare_v2(name):
    df, graph_ids, g, added = raw[name]
    v = g.values.astype(np.float32); T, N = v.shape
    observed = v > 0
    if STUCK_AS_MISSING:
        observed &= ~stuck_mask(v)
    n_tr, n_va, n_te = split_sizes(T - SEQ_IN - SEQ_OUT + 1)
    train_last_row = (SEQ_IN - 1 + n_tr - 1) + SEQ_OUT                 # last raw row any training target uses
    hist = history_profile(v, observed, g.index, train_last_row)
    filled, short, long_ = fill_inputs(v, observed, hist)
    tod = ((g.index.hour * 60 + g.index.minute) / 1440).values.astype(np.float32)
    x, y, xm, anchors, x_off, y_off = build_samples(filled, v, observed, tod)
    assert len(anchors) == n_tr + n_va + n_te
    parts = {'train': slice(0, n_tr), 'val': slice(n_tr, n_tr + n_va), 'test': slice(n_tr + n_va, None)}
    out_dir = os.path.join(OUT_ROOT, name)
    summ = dict(dataset=name, sensors=N, timestamps=T, rows_added_by_regrid=len(added),
                train_last_raw_row=int(train_last_row), train_last_raw_time=str(g.index[train_last_row]),
                settings=dict(SHORT_GAP=SHORT_GAP, STUCK_STEPS=STUCK_STEPS, STUCK_AS_MISSING=STUCK_AS_MISSING,
                              ADD_MASK_CHANNEL=ADD_MASK_CHANNEL),
                inputs_filled_short=float(short.mean()), inputs_filled_long=float(long_.mean()),
                scaler_mean_observed=float(v[:train_last_row + 1][observed[:train_last_row + 1]].mean()),
                scaler_std_observed=float(v[:train_last_row + 1][observed[:train_last_row + 1]].std()),
                splits={})
    for part, sl in parts.items():
        a = anchors[sl]
        np.savez_compressed(os.path.join(out_dir, f'{part}.npz'),
                            x=x[sl], y=y[sl], x_mask=xm[sl].astype(np.uint8),
                            x_offsets=x_off.reshape(-1, 1), y_offsets=y_off.reshape(-1, 1),
                            t_anchor=g.index.values[a].astype('datetime64[ns]'), t_index=a)
        summ['splits'][part] = dict(samples=int(len(a)), first_anchor=str(g.index[a[0]]), last_anchor=str(g.index[a[-1]]),
                                    inputs_filled=float(1 - xm[sl].mean()),
                                    masked_targets=float((y[sl][..., 0] == 0).mean()))
        print(f'  saved {part:5s} x {x[sl].shape}  filled inputs {1 - xm[sl].mean():.2%}  missing targets {(y[sl][..., 0] == 0).mean():.2%}')
    json.dump(summ, open(os.path.join(out_dir, 'summary.json'), 'w'), indent=2)
    del x, y, xm; gc.collect()
    return summ

summ_v2 = {}
for name in DATASETS:
    done = all(os.path.exists(os.path.join(OUT_ROOT, name, f'{p}.npz')) for p in ['train', 'val', 'test', 'summary'] if p != 'summary') \
           and os.path.exists(os.path.join(OUT_ROOT, name, 'summary.json'))
    if done and not REBUILD:
        summ_v2[name] = json.load(open(os.path.join(OUT_ROOT, name, 'summary.json')))
        print(f'{name}: fixed files already in Drive, skipping (set REBUILD = True to redo)')
        continue
    print(f'==== {name} ====')
    summ_v2[name] = prepare_v2(name)

METR-LA: fixed files already in Drive, skipping (set REBUILD = True to redo)


## 6. Check the saved files (every line must say OK)

In [7]:
for name in DATASETS:
    print(f'==== {name} ====')
    d2 = os.path.join(OUT_ROOT, name); d1 = os.path.join(V1_ROOT, name)
    f2 = {p: np.load(os.path.join(d2, f'{p}.npz')) for p in ['train', 'val', 'test']}
    _, graph_ids, g, added = raw[name]
    N = len(graph_ids)
    for p in f2:
        assert f2[p]['x'].shape[1:3] == (SEQ_IN, N) and f2[p]['y'].shape[1:] == (SEQ_OUT, N, 2)
    print('OK  shapes')
    assert all((f2[p]['x'][..., 0] > 0).all() for p in f2);                       print('OK  no zeros left in the inputs')
    ta = [f2[p]['t_anchor'] for p in f2]
    assert ta[0][-1] < ta[1][0] and ta[1][-1] < ta[2][0];                          print('OK  train < val < test in time')
    # targets are the raw readings, never filled
    for p in f2:
        for k in [0, len(f2[p]['y']) // 2, len(f2[p]['y']) - 1]:
            t = int(f2[p]['t_index'][k])
            assert np.allclose(f2[p]['y'][k, :, :, 0], g.values[t + 1:t + SEQ_OUT + 1])
    print('OK  targets are the untouched raw readings (missing stay 0 and stay masked)')
    # observed inputs are unchanged
    x = f2['test']['x'][..., 0]; m = f2['test']['x_mask'].astype(bool)
    t0 = int(f2['test']['t_index'][0])
    raw_win = g.values[t0 - SEQ_IN + 1:t0 + 1]
    assert np.allclose(x[0][m[0]], raw_win[m[0]]);                                  print('OK  real readings in the inputs are unchanged')
    if not added.size:
        f1 = np.load(os.path.join(d1, 'test.npz'))
        assert np.array_equal(f1['t_anchor'], f2['test']['t_anchor']);             print('OK  same test anchors as step 2')
    else:
        print(f'NOTE {len(added)} timestamps were added by re-gridding, so sample counts differ slightly from step 2')

==== METR-LA ====
OK  shapes
OK  no zeros left in the inputs
OK  train < val < test in time
OK  targets are the untouched raw readings (missing stay 0 and stay masked)
OK  real readings in the inputs are unchanged
OK  same test anchors as step 2


## 7. Step 2 vs step 2b: how much changed

In [8]:
rows = []
for name in DATASETS:
    s = summ_v2[name]
    v1 = json.load(open(os.path.join(V1_ROOT, name, 'summary.json')))
    rows.append(dict(dataset=name,
                     inputs_changed_train=f"{s['splits']['train']['inputs_filled']:.2%}",
                     inputs_changed_test=f"{s['splits']['test']['inputs_filled']:.2%}",
                     short_fills=f"{s['inputs_filled_short']:.2%}", long_fills=f"{s['inputs_filled_long']:.2%}",
                     scaler_v1=f"{v1.get('scaler_mean', float('nan')):.2f}" if 'scaler_mean' in v1 else 'with zeros',
                     scaler_v2=f"{s['scaler_mean_observed']:.2f} ± {s['scaler_std_observed']:.2f}",
                     test_targets_missing=f"{s['splits']['test']['masked_targets']:.2%}",
                     timestamps_added=s['rows_added_by_regrid']))
display(pd.DataFrame(rows).set_index('dataset'))
print('Targets are identical to step 2, so any score difference after retraining comes only from the inputs.')

,inputs_changed_train,inputs_changed_test,short_fills,long_fills,scaler_v1,scaler_v2,test_targets_missing,timestamps_added
dataset,,,,,,,,
METR-LA,7.13%,12.11%,0.81%,7.30%,with zeros,58.58 ± 12.83,12.11%,0


Targets are identical to step 2, so any score difference after retraining comes only from the inputs.


# Part B: retrain on the fixed data

## 7. Training settings

* `VERSIONS` – the graph versions to compare. `original` (no added links), `random` (25 random links), `express` (our 25 links).
* `SEEDS` – `[0]` = 3 runs. `[0, 1]` = 6 runs. Your old step-12 runs have seeds 0, 1, 2, so any of these can be compared.
* `SESSION_HOURS` – stop cleanly after this many hours (`None` = until done or Colab stops). Run all again to continue.
* Model settings are **the same as step 12** (taken from `code/train_gwnet.py` in Drive): 100 epochs, batch 64, learning rate 0.001, learned links off.

In [9]:
VERSIONS      = ['original', 'random', 'express']
SEEDS         = [0]
SESSION_HOURS = None
DATASET       = 'METR-LA'
RUNS_V2       = 'runs_v2'          # new folder; old runs stay in runs/
WORKER        = 'alzahraa'

import sys, importlib, glob
CODE_DIR = os.path.join(PROJECT_DIR, 'code')
assert os.path.exists(os.path.join(CODE_DIR, 'train_gwnet.py')), 'code/train_gwnet.py is missing (it is written by step 12, section 3)'
assert os.path.exists(os.path.join(CODE_DIR, 'metrics.py')), 'code/metrics.py is missing (step 9)'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import train_gwnet as T
T = importlib.reload(T)

# Point the training code at the FIXED data. Nothing else in the module changes.
V1_DIR = os.path.join(PROJECT_DIR, 'data', 'processed', DATASET)
V2_DIR = os.path.join(OUT_ROOT, DATASET)
T.data_dir = lambda project_dir, dataset: os.path.join(OUT_ROOT, dataset)
assert T.data_dir(PROJECT_DIR, DATASET) == V2_DIR

import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU, then Run all.'
print('GPU:', torch.cuda.get_device_name(0))
print('Settings (same as step 12):', T.DEFAULTS)
T.setup_graph_wavenet('/content/Graph-WaveNet')

JOBS = [dict(dataset=DATASET, graph=v, seed=s) for s in SEEDS for v in VERSIONS]
for j in JOBS:
    gf = T.graph_file(PROJECT_DIR, DATASET, j['graph'])
    assert os.path.exists(gf), f'graph file missing: {gf}'
print(f'{len(JOBS)} runs:', ', '.join(f"{j['graph']}/seed{j['seed']}" for j in JOBS))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Settings (same as step 12): {'epochs': 100, 'patience': None, 'batch_size': 64, 'lr': 0.001, 'weight_decay': 0.0001, 'dropout': 0.3, 'nhid': 32, 'adj_type': 'doubletransition', 'adaptive_adj': False}
3 runs: original/seed0, random/seed0, express/seed0


## 8. Status before training

If Colab died during a run, that run still carries your name for up to 30 minutes. The cell below hands it back to you after 10 minutes without a heartbeat, so it continues at once.

In [10]:
for j in JOBS:
    rd = T.run_dir(PROJECT_DIR, j['dataset'], j['graph'], j['seed'], RUNS_V2)
    c = T.read_json(os.path.join(rd, 'claim.json'), {})
    if c and c.get('worker') == WORKER and time.time() - c.get('heartbeat', 0) > 10 * 60:
        T.release_claim(rd); print('took back', os.path.relpath(rd, PROJECT_DIR))

board = T.status_board(PROJECT_DIR, JOBS, RUNS_V2)
display(board[['graph', 'seed', 'status', 'epoch', 'best_val_mae']])
est, total_h, _ = T.time_estimate(board, workers=1)
print(f'GPU time left: about {total_h:.1f} h' + ('' if (est['source'] == 'measured').all() else ' (T4 guess until the first epoch is timed)'))

,graph,seed,status,epoch,best_val_mae
0,original,0,done,100,2.743274
1,random,0,done,100,2.736518
2,express,0,done,100,2.737472


GPU time left: about 0.0 h


## 9. Train

Each epoch prints the time left in that run. Scoring is done in section 10 (so old and new runs are scored the same way).

In [11]:
n_done = T.work(PROJECT_DIR, JOBS, WORKER, order=1, runs_subdir=RUNS_V2,
                session_hours=SESSION_HOURS, score=False)
print(f'\nfinished {n_done} run(s) in this session')
board = T.status_board(PROJECT_DIR, JOBS, RUNS_V2)
display(board[['graph', 'seed', 'status', 'epoch', 'best_val_mae']])

All runs are done.

finished 0 run(s) in this session


,graph,seed,status,epoch,best_val_mae
0,original,0,done,100,2.743274
1,random,0,done,100,2.736518
2,express,0,done,100,2.737472


# Part C: compare

## 10. Old data vs fixed data

Every run, old and new, is scored by **step 9's `metrics.py` against the original step-2 test set** (`data/processed/METR-LA/test.npz`). The test targets and the jam list are identical in both, so the **only** difference between the two rows of a version is the input data the model was trained and tested with.

* `mae_60` – average error 60 minutes ahead (mph, lower is better).
* `mae_60_behind` – the same, only for the roads behind the bottleneck (step 5).
* `jam_recall_60` / `jam_precision_60` – share of real jams warned about 60 minutes ahead / share of warnings that were right.

**How to read it.** In step 12 the seed-to-seed spread of `mae_60` was 0.01–0.03 mph. A difference smaller than that is noise.

In [12]:
import metrics as M
M = importlib.reload(M)
COLS = ['mae_15', 'mae_30', 'mae_60', 'mae_60_behind', 'mae_avg', 'jam_recall_60', 'jam_precision_60']

rows = []
for v in VERSIONS:
    for s in [0, 1, 2]:
        for data_tag, sub in [('old (step 2)', 'runs'), ('fixed (step 2b)', RUNS_V2)]:
            rd = T.run_dir(PROJECT_DIR, DATASET, v, s, sub)
            if not os.path.exists(os.path.join(rd, 'test_predictions.npz')):
                continue
            r = M.score_run(rd, V1_DIR, save=(sub == RUNS_V2))       # never rewrites anything in old runs
            rows.append(dict(data=data_tag, graph=v, seed=s, **{c: r.get(c, np.nan) for c in COLS}))
cmp_ = pd.DataFrame(rows)
assert len(cmp_[cmp_['data'] == 'fixed (step 2b)']), 'No finished run on the fixed data yet: run section 9 first.'
cmp_.to_csv(os.path.join(PROJECT_DIR, RUNS_V2, 'compare_old_vs_fixed.csv'), index=False)

# only seeds that exist on BOTH data versions, so the comparison is like for like
fixed_seeds = sorted(int(x) for x in cmp_[cmp_['data'] == 'fixed (step 2b)']['seed'].unique())
paired = cmp_[cmp_['seed'].isin(fixed_seeds)]
print(f'Seeds compared: {fixed_seeds}')
tab = paired.groupby(['graph', 'data'])[COLS].mean().reindex(VERSIONS, level=0)
display(tab.round(3))

print('\nPer seed, 60-min MAE (mph):')
display(paired.pivot_table(index=['seed', 'graph'], columns='data', values='mae_60').reindex(VERSIONS, level=1).round(3))

Seeds compared: [0]


mae_15  mae_30  mae_60  mae_60_behind  mae_avg  \
graph    data                                                              
original fixed (step 2b)   2.691   3.071   3.520          2.416    3.034   
         old (step 2)      2.716   3.106   3.587          2.465    3.072   
random   fixed (step 2b)   2.686   3.064   3.521          2.402    3.035   
         old (step 2)      2.725   3.119   3.573          2.439    3.078   
express  fixed (step 2b)   2.685   3.057   3.496          2.423    3.019   
         old (step 2)      2.713   3.099   3.565          2.449    3.066   

                          jam_recall_60  jam_precision_60  
graph    data                                              
original fixed (step 2b)          0.102             0.483  
         old (step 2)             0.083             0.531  
random   fixed (step 2b)          0.111             0.508  
         old (step 2)             0.093             0.557  
express  fixed (step 2b)          0.059             0.506  
         old (step 2)             0.097             0.564


Per seed, 60-min MAE (mph):


data           fixed (step 2b)  old (step 2)
seed graph                                  
0    original            3.520         3.587
     random              3.521         3.573
     express             3.496         3.565

In [13]:
# The answer
NOISE = 0.03     # mph: seed-to-seed spread of mae_60 in step 12
p = paired.pivot_table(index='seed', columns=['data', 'graph'], values='mae_60')

def gap(data_tag, a, b):            # positive = a is better (lower error) than b
    if (data_tag, a) not in p.columns or (data_tag, b) not in p.columns:
        return np.nan, np.array([])
    d = (p[(data_tag, b)] - p[(data_tag, a)]).dropna()
    return d.mean(), d.values

missing = [(t, v) for t in ['old (step 2)', 'fixed (step 2b)'] for v in VERSIONS if (t, v) not in p.columns]
if missing:
    print('Missing runs (no answer possible for these until they exist):', missing)

lines = []
for tag in ['old (step 2)', 'fixed (step 2b)']:
    g_r, per_r = gap(tag, 'express', 'random')
    g_o, per_o = gap(tag, 'express', 'original')
    lines.append(f'{tag:16s}: express beats random by {g_r:+.3f} mph, original by {g_o:+.3f} mph '
                 f'(per seed vs random: {np.round(per_r, 3).tolist()})')
print('\n'.join(lines))

all_gain = (p['old (step 2)'] - p['fixed (step 2b)']).mean().dropna()
print('\nHow much the data fix alone lowered the 60-min error, per version:')
print(all_gain.round(3).to_string())

g_old, _ = gap('old (step 2)', 'express', 'random')
g_new, _ = gap('fixed (step 2b)', 'express', 'random')
print()
if np.isnan(g_old) or np.isnan(g_new):
    print('ANSWER: not yet. express and random are needed on both the old and the fixed data (see "Missing runs").')
elif g_new - g_old > NOISE and g_new > NOISE:
    print(f'ANSWER: the fix MATTERS. On fixed data express beats random by {g_new:.3f} mph (was {g_old:.3f}). '
          'Worth retraining the other versions and seeds on the fixed data.')
else:
    print(f'ANSWER: the fix does NOT explain the tie. On the fixed data express vs random is {g_new:+.3f} mph '
          f'(old data: {g_old:+.3f}), not a win bigger than the {NOISE} mph noise. '
          'The cause is the graph setup, not the missing data.')
if len(fixed_seeds) == 1:
    print('(1 seed only. If the answer is close to the noise line, add seed 1 in section 7 and Run all.)')

old (step 2)    : express beats random by +0.007 mph, original by +0.022 mph (per seed vs random: [0.007])
fixed (step 2b) : express beats random by +0.025 mph, original by +0.024 mph (per seed vs random: [0.025])

How much the data fix alone lowered the 60-min error, per version:
graph
express     0.069
original    0.068
random      0.051

ANSWER: the fix does NOT explain the tie. On the fixed data express vs random is +0.025 mph (old data: +0.007), not a win bigger than the 0.03 mph noise. The cause is the graph setup, not the missing data.
(1 seed only. If the answer is close to the noise line, add seed 1 in section 7 and Run all.)
